In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

### Example

An institute for public policy in Washington, D.C., hires a number of college interns every summer. This year, Sara Anderson, a third-year Economics major from Massachusetts, is selected as one of the research interns. Her first assignment is to conduct data analysis to help congressional offices gain a better understanding of U.S. residents whose incomes are below the poverty level.

To complete her assignment, Sara extracts a relevant data set maintained by the U.S. Census Bureau.

The data set has 9,980 observations, each observation contains an individual's marital status (**Married**; yes/no), **sex** (Female; yes/no), **ethnicity** (White; yes/no), **age groups** (Age: 1 for [18, 25), 2 for [25, 35), 3 for [35, 45), 4 for [45, 55), and 5 for 55 years and older), whether or not the individual receives college-level education (**Edu**; yes/no), and whether or not the individual's income is below the poverty level (**Poverty** = 1 if living in poverty, 0 otherwise).


A Portion of the Census Data |
| --- |

| a. Census_Data Worksheet |
| --- |

| Married | Female | White | Age | Edu | Poverty |
| --- | --- | --- | --- | --- | --- |
| Y | N | Y | 1 | N | 0 |
| Y | N | Y | 1 | N | 0 |
| ... | ... | ... | ... | ... | ... |
| N | Y | N | 5 | N | 1 |



Help Sara develop a naive Bayes classification model to classify whether or not an individual is likely to live in poverty using all five predictor variables. Report the relevant performance measures and score new observations.

#### Solution

As before, we will use the scikit-learn (sklearn) package for classification with *k*-fold cross-validation. So, we will divide the data into two partitions for training and testing.

Import the ***census_Data*** data into a Pandas DataFrame (table) and label it myData.

In [2]:
import pandas as pd
data = pd.read_csv("/content/census_data.csv")

In [3]:
data.head()

,Married,Female,White,Age,Edu,Poverty
0,Y,N,Y,1,N,0
1,Y,N,Y,1,N,0
2,Y,N,Y,1,Y,0
3,N,N,Y,1,Y,1
4,N,N,N,1,N,1


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9980 entries, 0 to 9979
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Married  9980 non-null   object
 1   Female   9980 non-null   object
 2   White    9980 non-null   object
 3   Age      9980 non-null   int64 
 4   Edu      9980 non-null   object
 5   Poverty  9980 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 467.9+ KB


As shown in the output above, each of the variables could be treated as categorical data (i.e., since each column consists of two possible values such as yes/no or 1/0). Thus, we can convert all the columns to categorical data using the Pandas **astype** method. Enter:

In [5]:
data = data.astype('category')

To check the result, we can use the **dtypes** property:

In [6]:
data.dtypes

,0
Married,category
Female,category
White,category
Age,category
Edu,category
Poverty,category


Partition the data into X and y

In [7]:
X = data.drop(columns = ['Poverty'])
y = data['Poverty']

In [8]:
X.head()

,Married,Female,White,Age,Edu
0,Y,N,Y,1,N
1,Y,N,Y,1,N
2,Y,N,Y,1,Y
3,N,N,Y,1,Y
4,N,N,N,1,N


In [9]:
y.head()

,Poverty
0,0
1,0
2,0
3,1
4,1


### train_test_split

Next we divide the data in training (70\%) and testing (30\%) sets. Enter:

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

### OrdinalEncode

Before we begin analyzing the data using the categorical naive bayes algorithm, we need to encode the predictors (which may also be referred to as features) as integer arrays for our model. To do this, we use scikit-learn's **OrdinalEncoder** function and **fit** the scaler with our training variables (i.e., X_train). We subsequently store the fitted scaler in a variable with the label train_encoder.

**Note:** Naive Bayes and Tree based algorithm deal directly with joint frequency distribution which means they deal directly with categorical data without OneHotEncoding. We only need to present the categories in numerical form.

In [11]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()

train_encoder = encoder.fit(X_train)

In [12]:
train_encoder

OrdinalEncoder()

Next, we apply our scaler based on the training variables to both the train and testing data using the **transform** method. It is important to based our scaling exclusively on the training data variables **to avoid data leakage** across train and test. Enter:

In [13]:
X_train_enc = train_encoder.transform(X_train)
X_test_enc = train_encoder.transform(X_test)

In [14]:
X_train_enc

array([[0., 1., 1., 0., 0.],
       [0., 1., 1., 2., 1.],
       [0., 0., 1., 0., 0.],
       ...,
       [0., 0., 1., 2., 0.],
       [0., 0., 1., 0., 0.],
       [1., 0., 0., 3., 0.]])

## **CategoricalNB**
We set up our naive Bayes model using scikit-learn. Specifically, because all of our predictor variables consist of categorical data as noted above, we use the **CategoricalNB** function (for details on other types of naive Bayes models in scikit-learn, consult the [User Guide](https://scikit-learn.org/stable/modules/naive_bayes.html).

In [15]:
from sklearn.naive_bayes import CategoricalNB

In [16]:
cnb = CategoricalNB()

In [17]:
cnb

CategoricalNB()

### GridSearchCV

To evaluate with 10-fold cross-validation, we again use the scikit-learn **GridSearchCV** function as we did previously.

Because GridSearchCV is designed to evaluate various model settings (which are also referred to as tuning parameters or hyperparameters), we must specify those as well by supplying relevant parameters based on the **CategoricalNB** function's [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.CategoricalNB.html)).

We enter the parameters we intend to evaluate into a dictionary with the label paramGrid; in this case, we explore how accuracy is influenced by whether `fit_prior` is set to `True` or `False`, which influences how class prior probabilities are calculated.



In [18]:
param_grid = {'fit_prior': [True, False]}

Next, we use paramGrid in our 10-fold **GridSearchCV** procedure with *cv* set to 10 and *scoring* set to `'accuracy'`

In [19]:
from sklearn.model_selection import GridSearchCV

In [20]:
search = GridSearchCV(cnb, param_grid=param_grid, cv = 10, scoring = 'accuracy')

In [21]:
search

GridSearchCV(cv=10, estimator=CategoricalNB(),
             param_grid={'fit_prior': [True, False]}, scoring='accuracy')

Finally, we conduct the actual search using the **fit** method and use our relevant training data.

In [22]:
search_fit = search.fit(X_train_enc, y_train)

In [23]:
search_fit

GridSearchCV(cv=10, estimator=CategoricalNB(),
             param_grid={'fit_prior': [True, False]}, scoring='accuracy')

To see the results of our modeling, we use the **cv_results_** property of searchFit. Enter:

In [24]:
search_fit.cv_results_

{'mean_fit_time': array([0.00796106, 0.00770462]),
 'std_fit_time': array([0.00198661, 0.0029105 ]),
 'mean_score_time': array([0.00320861, 0.00366716]),
 'std_score_time': array([0.00082349, 0.00193134]),
 'param_fit_prior': masked_array(data=[True, False],
              mask=[False, False],
        fill_value=True),
 'params': [{'fit_prior': True}, {'fit_prior': False}],
 'split0_test_score': array([0.79685265, 0.73104435]),
 'split1_test_score': array([0.80257511, 0.73676681]),
 'split2_test_score': array([0.82260372, 0.76108727]),
 'split3_test_score': array([0.78826896, 0.73247496]),
 'split4_test_score': array([0.78254649, 0.73247496]),
 'split5_test_score': array([0.80114449, 0.75822604]),
 'split6_test_score': array([0.78080229, 0.73925501]),
 'split7_test_score': array([0.78653295, 0.72492837]),
 'split8_test_score': array([0.77363897, 0.70630372]),
 'split9_test_score': array([0.80515759, 0.75358166]),
 'mean_test_score': array([0.79401232, 0.73761432]),
 'std_test_score': ar

In [25]:
pd.DataFrame(search_fit.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_fit_prior,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.007961,0.001987,0.003209,0.000823,True,{'fit_prior': True},0.796853,0.802575,0.822604,0.788269,0.782546,0.801144,0.780802,0.786533,0.773639,0.805158,0.794012,0.013726,1
1,0.007705,0.002911,0.003667,0.001931,False,{'fit_prior': False},0.731044,0.736767,0.761087,0.732475,0.732475,0.758226,0.739255,0.724928,0.706304,0.753582,0.737614,0.015721,2


To focus on the key outcomes, we sort by rank_test_score and select only the relevant columns for display using Pandas. Enter:

In [26]:
pd.DataFrame(search_fit.cv_results_).sort_values('rank_test_score')[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]

,params,mean_test_score,std_test_score,rank_test_score
0,{'fit_prior': True},0.794012,0.013726,1
1,{'fit_prior': False},0.737614,0.015721,2


As shown in the output above, the best average accuracy for the model with optimal parameter`fit_prior:True`, based on the *training data*.

We can generate predictions on the *test data* using the **predict** method of our searchFit. Enter:

In [27]:
y_pred = search_fit.predict(X_test_enc)

In [28]:
y_pred

array([1, 1, 1, ..., 1, 1, 1])

In [29]:
y_test[:10]

,Poverty
1253,1
8224,1
1658,1
7733,1
1441,1
1056,1
35,1
7150,0
1662,1
4047,1


In [30]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.8002672010688042

Moreover, we can also generate additional types of output and visualizations to explore the results further.

In [31]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=['Poverty:0','Poverty:1']))

              precision    recall  f1-score   support

   Poverty:0       0.54      0.59      0.56       656
   Poverty:1       0.88      0.86      0.87      2338

    accuracy                           0.80      2994
   macro avg       0.71      0.72      0.72      2994
weighted avg       0.81      0.80      0.80      2994



### Multinomial NB

In [32]:
from sklearn.naive_bayes import MultinomialNB

In [33]:
mnb = MultinomialNB()

In [34]:
grid_mnb = GridSearchCV(mnb, param_grid=param_grid, cv = 10, scoring = 'accuracy')

In [35]:
grid_mnb.fit(X_train_enc, y_train)

GridSearchCV(cv=10, estimator=MultinomialNB(),
             param_grid={'fit_prior': [True, False]}, scoring='accuracy')

In [36]:
y_pred_mnb = grid_mnb.predict(X_test_enc)

In [37]:
print(classification_report(y_test, y_pred_mnb, target_names=['Poverty:0','Poverty:1']))

              precision    recall  f1-score   support

   Poverty:0       0.00      0.00      0.00       656
   Poverty:1       0.78      1.00      0.88      2338

    accuracy                           0.78      2994
   macro avg       0.39      0.50      0.44      2994
weighted avg       0.61      0.78      0.68      2994

